# Verificación de Ollama en SageMaker

Comprueba que Docker usa el disco persistente, descarga el modelo, registra su nombre y versión y prueba una instrucción.

In [ ]:
!docker info --format 'data-root: {{.DockerRootDir}}'
!docker ps --filter name=ollama
!tail -n 20 /home/ec2-user/SageMaker/logs/on-start.log

In [ ]:
MODEL = "llama3.2:3b"  # cabe en m5.xlarge (4 vCPU, 16 GB RAM, sin GPU)
!docker exec ollama ollama pull {MODEL}

In [ ]:
import json, requests

OLLAMA = "http://localhost:11434"
version = requests.get(f"{OLLAMA}/api/version").json()["version"]
tags = requests.get(f"{OLLAMA}/api/tags").json()["models"]
info = next(m for m in tags if m["name"] == MODEL)

registro = {
    "ollama_version": version,
    "modelo": info["name"],
    "digest": info["digest"],
    "tamano_bytes": info["size"],
    "detalles": info["details"],
}
with open("modelo.json", "w") as f:
    json.dump(registro, f, indent=2)
registro

In [ ]:
r = requests.post(f"{OLLAMA}/api/generate", json={
    "model": MODEL,
    "prompt": "Responde en una frase: ¿qué es un dinosaurio?",
    "stream": False,
})
print(r.json()["response"])